<a href="https://colab.research.google.com/github/dsustana/steam-spotlight/blob/main/steam_spotlight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Steam Spotlight
My app helps Steam's store team decide which titles it should feature on the website landing page by analyzing the critic reviews, required age, and average playtime hours of each title.

# Before use, run the below setup cell.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/Colab Notebooks/


# Sub-Question 1
Is the game in question family-friendly enough to be featured to users whose ages are not verified? Ratings of 0 are marked for further review. (*required_age* column)

In [ ]:
def check_age_suitability(games):
    results = games[["name", "required_age"]].copy()
    family_friendly = []

    for age in results["required_age"]:
        if age == 0:
            family_friendly.append("Needs further review")
        elif age > 0 and age <= 16:
            family_friendly.append("Family friendly")
        elif age > 16:
            family_friendly.append("Not family friendly")
        else:
            family_friendly.append("Age information missing or invalid")

    results["family_friendly"] = family_friendly
    return results


my_function = check_age_suitability
my_result = my_function(games)
print(my_result.head(20))

                               name  required_age       family_friendly
0                  Counter-Strike 2             0  Needs further review
1               PUBG: BATTLEGROUNDS             0  Needs further review
2                            Dota 2             0  Needs further review
3         Grand Theft Auto V Legacy            17   Not family friendly
4   Tom Clancy's Rainbow Six® Siege            17   Not family friendly
5                   Team Fortress 2             0  Needs further review
6                          Terraria             0  Needs further review
7                              Rust             0  Needs further review
8                       Garry's Mod             0  Needs further review
9                     Apex Legends™             0  Needs further review
10               Black Myth: Wukong            13       Family friendly
11         The Witcher 3: Wild Hunt            17   Not family friendly
12                       ELDEN RING            16       Family f

# Sub-Question 2
What do critics say about the game? (*critic_reviews* column; evaluated by Google Gemini via an LLM API)

In [ ]:
from google import genai
from google.colab import userdata

key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)
MODEL = "gemini-flash-lite-latest"


def summarize_critic_reviews(games, game_name):
    selected = games[games["name"] == game_name]

    for quotes in selected["critic_reviews"]:
        if type(quotes) == str and quotes.strip() != "":
            print("Critic quotes for", game_name)
            print(quotes)

            prompt = f"What do critics say about {game_name}? Write one sentence for Steam's store team. "
            prompt = prompt + f"Include only praise or criticism supported by these quotes:\n{quotes}"
            response = client.models.generate_content(model=MODEL, contents=prompt)
            return response.text.strip()

    return "No critic quotes found for this game."


summary = summarize_critic_reviews(games, "Dota 2")
print("Review summary:", summary)

# Sub-Question 3
How long do users typically spend on each game relative to its counterparts in the same genre? (*avg_playtime_hours* column; includes a class)

In [ ]:
class GenreGroup:
    def __init__(self, games, genre):
        self.genre = genre
        self.games = games[games["genres"].apply(self.has_genre)]

    def has_genre(self, genres):
        return type(genres) == str and self.genre in genres.split(", ")

    def compare_playtime(self):
        results = self.games[["name", "avg_playtime_hours"]].copy()
        results["genre"] = self.genre
        peer_count = len(results) - 1
        results["peer_games"] = peer_count

        if peer_count > 0:
            hours = results["avg_playtime_hours"]
            peer_averages = (hours.sum() - hours) / peer_count
            results["peer_avg_hours"] = peer_averages.round(1)
            results["hours_vs_peers"] = (hours - peer_averages).round(1)
        else:
            results["peer_avg_hours"] = None
            results["hours_vs_peers"] = None

        return results


strategy_games = GenreGroup(games, "Strategy")
playtime_results = strategy_games.compare_playtime()
print(playtime_results[playtime_results["name"] == "Dota 2"])